In [87]:
using LowLevelFEM, LinearAlgebra, SparseArrays

In [88]:
line_mesh(n=2, order=2)

mat = Material("body")
Pu = Problem([mat], type=:VectorField, dim=2, field=:u, rhs_field=:f)
Pφ = Problem([mat], type=:ScalarField, dim=2, field=:φ, rhs_field=:m, reducedOrder=false)

Problem("line_mesh", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :m, false)

In [89]:
a = 80
v = 2
E = mat.E
G = mat.μ
Iz = a^4 / 12 - (a - 2v)^4 / 12
κ = 2 / 3
A = a * a - (a - 2v)^2

624

In [90]:
t = tangentVector(Pu, "body")
tx, ty, tz = t[1], t[2], t[3]

nx = -ty
ny = tx
nz = 0

0

In [91]:
showElementResults(t)
showElementResults(VectorField([nx, ny, 0ny]))

1

In [92]:
a = [tx * tx tx * ty ty * tx ty * ty]   # axial projector
s = [nx * tx nx * ty ny * tx ny * ty]   # shear projector

1×4 Matrix{ScalarField}:
 ScalarField([[-0.0; -0.0; -0.0;;], [-0.0; -0.0; -0.0;;]], Matrix{Float64}(undef, 0, 0), [0.0], [3, 4], 1, :scalar, Problem("line_mesh", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 1.15385e5, 76923.1, 1.66667e5, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false))  …  ScalarField([[0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0;;]], Matrix{Float64}(undef, 0, 0), [0.0], [3, 4], 1, :scalar, Problem("line_mesh", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 1.15385e5, 76923.1, 1.66667e5, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false))

In [93]:
Cuu = E * A * (a' * a) + κ * G * A * (s' * s)

Cuφ = -κ * G * A * s'   # 4×1
Cφu = -κ * G * A * s    # 1×4

1×4 Matrix{ScalarField}:
 ScalarField([[0.0; 0.0; 0.0;;], [0.0; 0.0; 0.0;;]], Matrix{Float64}(undef, 0, 0), [0.0], [3, 4], 1, :scalar, Problem("line_mesh", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 1.15385e5, 76923.1, 1.66667e5, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false))  …  ScalarField([[-0.0; -0.0; -0.0;;], [-0.0; -0.0; -0.0;;]], Matrix{Float64}(undef, 0, 0), [0.0], [3, 4], 1, :scalar, Problem("line_mesh", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 1.15385e5, 76923.1, 1.66667e5, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false))

In [94]:
Kuu = ∫(Grad(Pu) ⋅ Cuu ⋅ Grad(Pu); Γ="body")

Kuφ = ∫(Grad(Pu) ⋅ Cuφ ⋅ Id(Pφ); Γ="body")
Kφu = ∫(Id(Pφ) ⋅ Cφu ⋅ Grad(Pu); Γ="body")

Kφφ = ∫(Id(Pφ) ⋅ (κ * G * A) ⋅ Id(Pφ); Γ="body") +
      ∫(AxialGrad(Pφ) ⋅ (E * Iz) ⋅ AxialGrad(Pφ); Γ="body")

sparse([1, 3, 4, 2, 3, 5, 1, 2, 3, 4, 5, 1, 3, 4, 2, 3, 5], [1, 1, 1, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 5, 5, 5], [5.909439999999999e11, 8.441973333333327e10, -6.753610666666665e11, 5.909439999999999e11, 8.441973333333327e10, -6.753610666666664e11, 8.441973333333327e10, 8.441973333333327e10, 1.1818879999999998e12, -6.753610666666663e11, -6.753610666666664e11, -6.753610666666665e11, -6.753610666666663e11, 1.3507327999999993e12, -6.753610666666664e11, -6.753610666666664e11, 1.3507327999999993e12], 5, 5)

In [95]:
K = SystemMatrix([Kuu Kuφ; Kφu Kφφ])

sparse([1, 5, 7, 2, 6, 8, 11, 13, 14, 3  …  6, 8, 11, 13, 14, 4, 6, 12, 13, 15], [1, 1, 1, 2, 2, 2, 2, 2, 2, 3  …  14, 14, 14, 14, 14, 15, 15, 15, 15, 15], [5.823999999999999e8, 8.319999999999991e7, -6.655999999999998e8, 1.4933333333333328e8, 2.133333333333331e7, -1.7066666666666663e8, 1.5999999999999994e7, -5.33333333333333e6, 2.133333333333333e7, 5.823999999999998e8  …  -2.133333333333333e7, -1.862645149230957e-9, -6.753610666666665e11, -6.753610666666663e11, 1.3507327999999993e12, -2.133333333333333e7, 2.133333333333333e7, -6.753610666666664e11, -6.753610666666664e11, 1.3507327999999993e12], 15, 15)

In [96]:
supp_u = BoundaryCondition("left", problem=Pu, ux=0, uy=0)
supp_φ = BoundaryCondition("left", problem=Pφ, φ=0)

BoundaryCondition("left", Problem("line_mesh", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :m, false), Dict{Symbol, Union{Function, Number, ScalarField}}(:φ => 0))

In [97]:
fu = ∫(Pu ⋅ [0, -1], Γ="right")
fφ = ∫(Pφ ⋅ 0, Γ="right")
fu2 = ∫(Pu ⋅ [0.0, -0.0], Γ="right")

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [98]:
F = SystemVector([fu + fu2, fφ])

SystemVector([0.0; 0.0; … ; 0.0; 0.0;;], nothing, Problem[Problem("line_mesh", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false), Problem("line_mesh", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :m, false)], [0, 10])

In [99]:


Tφ, Rφ = reductionMatrices(Pφ)

Tu = spdiagm(0 => ones(Pu.non*Pu.pdim))
TT = blockdiag(Tu, Tφ)

Kr = TT' * K.A * TT

eigvals(Matrix(Kr))

13-element Vector{ComplexF64}:
 -9.482414482655161e-6 + 0.0im
 -4.703232445585583e-6 - 1.689447815906969e-6im
 -4.703232445585583e-6 + 1.689447815906969e-6im
   5.978082808784529e7 + 0.0im
   1.789433529868813e8 + 0.0im
  1.9312551437146682e8 + 0.0im
  4.4155057619272023e8 + 0.0im
   6.103851450155103e8 + 0.0im
   6.978920658878819e8 + 0.0im
  1.7204744856285188e9 + 0.0im
   2.380507934112086e9 + 0.0im
 2.5326613816866415e11 + 0.0im
  7.597877352623867e11 + 0.0im

In [100]:
Tφ, Rφ = reductionMatrices(Pφ)

norm(Rφ * Tφ - I, Inf)


0.0

In [101]:
size(Tφ)

(5, 3)

In [102]:
ndofs(P::Problem) = P.non * P.pdim

ndofs (generic function with 1 method)

In [103]:
Tφ, Rφ = reductionMatrices(Pφ)

Tu = spdiagm(0 => ones(ndofs(Pu)))
Ru = Tu

T = blockdiag(Tu, Tφ)
R = blockdiag(Ru, Rφ)

fixed = constrainedDoFs(K, [supp_u, supp_φ])
free = freeDoFs(K, [supp_u, supp_φ])
xD = zeros(size(R, 2))

free_r, fixed_r, xD_r =
    reduced_bc_data(R, fixed, xD[:, 1])

Kr = T' * K.A * T

λ = eigvals(Symmetric(Matrix(Kr[free_r, free_r])))
λ[1:10]

10-element Vector{Float64}:
 1.5527212597467056e7
 6.055930815640196e7
 1.3288831916157527e8
 3.9140064043680996e8
 5.182874291287973e8
 5.908398040633607e8
 1.5264706330090625e9
 2.3042826297057366e9
 9.674621887379579e10
 6.63052191816612e11

In [104]:
λ = eigvals(Symmetric(Matrix(Kr[free_r, free_r])))

λmax = maximum(abs, λ)
tol = 1e-12 * λmax

count(abs.(λ) .< tol)

0

In [105]:
#u, v, φ = solveField(K, F, support=[supp_u, supp_v, supp_φ, supp_u0, supp_v0, supp_φ0])
u, φ = solveField(K, F, support=[supp_u, supp_φ])

(VectorField(Matrix{Float64}[], [0.0; 0.0; … ; 0.0; -2.3439165772859875e-8;;], [0.0], Int64[], 1, :v2D, Problem("line_mesh", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false)), ScalarField(Matrix{Float64}[], [0.0; -3.948498938643498e-12; … ; -1.7274680258127389e-12; -3.701717495134488e-12;;], [0.0], Int64[], 1, :scalar, Problem("line_mesh", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 5, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :m, false)))

In [106]:
showDoFResults(fu, name="force")
u0 = showDoFResults(u, name="u", factor=1, visible=true)
φ0 = showDoFResults(φ, name="φ")

4

In [107]:
n = VectorField([nx, ny, 0nx])

elementwise VectorField
[[-0.0; 1.0; … ; 1.0; -0.0;;], [-0.0; 1.0; … ; 1.0; -0.0;;]]

In [108]:
g = grad(expandTo3D(u))

ε = t ⋅ (g * t)
γ = n ⋅ (g * t) - nodesToElements(φ)

κb = grad(φ) ⋅ t

N = E * A * ε
T = κ * G * A * γ
Mh = E * Iz * κb

elementwise ScalarField
[[-0.9999997367670175; -0.5000002632329869; -0.7500000000000022;;], [-0.4999997367670169; -2.632329848485181e-7; -0.250000000000001;;]]

In [109]:
N0 = showElementResults(N, name="N")
T0 = showElementResults(T, name="T")
Mh0 = showElementResults(Mh, name="Mh")

7

In [110]:
plotOnPath("body", N0)
plotOnPath("body", T0)
plotOnPath("body", Mh0)

10

In [111]:
openPostProcessor()